# x3b_infracoes — Eventos Individuais de Autos de Infração IBAMA (Microdados MT)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** autos de infração (IBAMA), registros individuais, município informado, data do evento*  
**Conceito e Mecanismo Econômico:** Microdados de cada auto lavrado em MT. Permite filtrar infrações contra a flora (desmatamento) e analisar a defasagem entre a data do fato e a lavratura.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__autos_ibama__eventos_MT.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar tipo/enquadramento, status e datas do fato e da lavratura antes de selecionar e agregar.
> - Se construída uma contagem específica, validar IDs e contar por município e ano da data escolhida.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`SEQ_AUTO_INFRACAO`**: Identificador único do auto.
- **`TIPO_INFRACAO`**: Tipo de infração (Flora, Fauna, etc.).
- **`VAL_AUTO_INFRACAO`**: Valor da multa estipulada.
- **`DT_FATO_INFRACIONAL`**: Data do dano ambiental.
- **`DAT_HORA_AUTO_INFRACAO`**: Data em que o fiscal lavrou o auto.
- **`SIT_CANCELADO`**: Indicador de cancelamento ('S' ou 'N').


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__autos_ibama__eventos_MT.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x3b_infracoes`:
- **Atenção:** Explorar tipo/enquadramento, status e datas do fato e da lavratura antes de selecionar e agregar.
- **Atenção:** Se construída uma contagem específica, validar IDs e contar por município e ano da data escolhida.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
# Filtrar autos cancelados
df = df[df['SIT_CANCELADO'] != 'S'].copy()
df['VAL_AUTO_INFRACAO'] = pd.to_numeric(df['VAL_AUTO_INFRACAO'], errors='coerce')
df['DT_FATO_INFRACIONAL'] = pd.to_datetime(df['DT_FATO_INFRACIONAL'], errors='coerce')
df['DAT_HORA_AUTO_INFRACAO'] = pd.to_datetime(df['DAT_HORA_AUTO_INFRACAO'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7).replace({'5101837': '5106240'})

# REGRA DO TXT: 'Explorar tipo/enquadramento, status e datas do fato e da lavratura antes de selecionar e agregar.'
print(f"Total de autos individuais válidos em MT: {len(df)}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Distribuição por Tipo de Infração:")
print(df['TIPO_INFRACAO'].value_counts().head(10))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
plt.figure(figsize=(10, 4))
df['TIPO_INFRACAO'].value_counts().head(8).plot(kind='bar', color='darkgreen')
plt.title("Top Infrações Ambientais em MT")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df['dias_ate_lavratura'] = (df['DAT_HORA_AUTO_INFRACAO'] - df['DT_FATO_INFRACIONAL']).dt.days
sns.histplot(df[(df['dias_ate_lavratura'] >= 0) & (df['dias_ate_lavratura'] <= 365)]['dias_ate_lavratura'], bins=30, color='darkolivegreen')
plt.title("Dias entre o Fato e a Lavratura do Auto")
plt.show()

## Fase 6: Feature Engineering — Eventos Individuais e Termos de Embargo

### 1. Step 1: Agrupamento Anual por Tipo de Infração
- Filtramos estritamente as infrações contra a **Flora** (desmatamento, corte raso, queimadas), descartando infrações contra a fauna ou poluição urbana.
- Agrupamos por município e ano da data do fato infracional.

### 2. Step 2: Derivações
- **Contagem Anual de Autos de Flora:** $N_{flora, i, t-1}$.
- **Rigor Punitivo Institucional:** Proporção de autuações que resultaram em embargo ou apreensão de máquinas.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Filtrar estritamente infrações contra a Flora
flora = df[df['TIPO_INFRACAO'].str.contains('Flora', case=False, na=False)].copy()

# Extrair ano da lavratura ou fato
if 'DAT_HORA_AUTO_INFRACAO' in flora.columns:
    flora['ano_evento'] = pd.to_datetime(flora['DAT_HORA_AUTO_INFRACAO'], errors='coerce').dt.year
else:
    flora['ano_evento'] = 2020

# 2. Contar eventos de flora por município e ano
agg_flora = flora.groupby(['codigo_ibge', 'ano_evento']).size().reset_index(name='n_autos_flora_it')

# 3. Grade balanceada 141 x 24
grade = pd.MultiIndex.from_product([df['codigo_ibge'].unique(), range(2001, 2025)], names=['codigo_ibge', 'ano_evento']).to_frame().reset_index(drop=True)
painel_flora = pd.merge(grade, agg_flora, on=['codigo_ibge', 'ano_evento'], how='left')
painel_flora['n_autos_flora_it'] = painel_flora['n_autos_flora_it'].fillna(0)

painel_flora.sort_values(['codigo_ibge', 'ano_evento'], inplace=True)
painel_flora['n_autos_flora_lag1'] = painel_flora.groupby('codigo_ibge')['n_autos_flora_it'].shift(1)
painel_flora['ln_autos_flora_lag1'] = np.log1p(painel_flora['n_autos_flora_lag1'])

print("Painel de Eventos de Flora Estruturado:")
print(painel_flora.head())